In [1]:
#câu 1
import keras
from keras import layers
from keras.datasets import cifar10
import numpy as np

# 1. Nạp và tiền xử lý
(x_train, _), (x_test, _) = cifar10.load_data()
x_train = x_train.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.
x_train = x_train.reshape((len(x_train), 3072))
x_test = x_test.reshape((len(x_test), 3072))

# 2. Xây dựng mô hình Multi-layers (3072 -> 512 -> 128 -> 32)
input_img = keras.Input(shape=(3072,))
encoded = layers.Dense(512, activation='relu')(input_img)
encoded = layers.Dense(128, activation='relu')(encoded)
encoded = layers.Dense(32, activation='relu')(encoded)

decoded = layers.Dense(128, activation='relu')(encoded)
decoded = layers.Dense(512, activation='relu')(decoded)
decoded = layers.Dense(3072, activation='sigmoid')(decoded)

autoencoder_cifar = keras.Model(input_img, decoded)
autoencoder_cifar.compile(optimizer='adam', loss='binary_crossentropy')

# 3. Huấn luyện (Yêu cầu epochs: 50, 100, 200)
autoencoder_cifar.fit(x_train, x_train, epochs=50, batch_size=256, shuffle=True, validation_data=(x_test, x_test))
autoencoder_cifar.save('cifar10_auto.h5')

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 26s 0us/step
Epoch 1/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 37s 177ms/step - loss: 0.6403 - val_loss: 0.6119
Epoch 2/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 30s 150ms/step - loss: 0.6057 - val_loss: 0.6033
Epoch 3/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 30s 151ms/step - loss: 0.6001 - val_loss: 0.5981
Epoch 4/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 30s 151ms/step - loss: 0.5960 - val_loss: 0.5964
Epoch 5/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 41s 149ms/step - loss: 0.5948 - val_loss: 0.5954
Epoch 6/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 28s 145ms/step - loss: 0.5932 - val_loss: 0.5939
Epoch 7/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 44s 159ms/step - loss: 0.5926 - val_loss: 0.5933
Epoch 8/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 29s 150ms/step - loss: 0.5922 - val_loss: 0.5930
Epoch 9/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 28s 144ms/step - loss: 0.5917 - val_loss: 0.5924
Epoch 10/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 31s 156ms/step - loss: 0.5910 - val_loss: 0.5919
Epoch 11/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 40s 152ms/step

In [10]:
#câu 2
import tensorflow as tf  # THÊM DÒNG NÀY ĐỂ HẾT LỖI
import tensorflow_datasets as tfds
import keras
from keras import layers

# 1. Tải bộ dữ liệu cats_vs_dogs (Lấy 20% để chạy cho nhanh)
print("--- Đang tải dữ liệu Cat vs Dog... ---")
data = tfds.load('cats_vs_dogs', split='train[:20%]', as_supervised=True)

# 2. Định nghĩa hàm tiền xử lý
def preprocess(img, label):
    # Resize về 64x64
    img = tf.image.resize(img, (64, 64))
    # Chuẩn hóa về [0, 1]
    img = tf.cast(img, tf.float32) / 255.0
    # Duỗi phẳng ảnh màu (64 * 64 * 3 = 12288)
    img = tf.reshape(img, (12288,))
    return img, img # Đầu ra o giống đầu vào x

# 3. Áp dụng map và tạo batch
train_data = data.map(preprocess).batch(32)

# 4. Xây dựng Autoencoder cho ảnh 64x64x3
# Encoder: Nén 12288 -> 512 -> 64
input_img = keras.Input(shape=(12288,))
encoded = layers.Dense(512, activation='relu')(input_img)
encoded = layers.Dense(64, activation='relu')(encoded) # Không gian ẩn s

# Decoder: Tái tạo 64 -> 512 -> 12288
decoded = layers.Dense(512, activation='relu')(encoded)
decoded = layers.Dense(12288, activation='sigmoid')(decoded) # Đầu ra o

auto_pet = keras.Model(input_img, decoded) # Mô hình o = D(E(x))
auto_pet.compile(optimizer='adam', loss='binary_crossentropy')

# 5. Huấn luyện (Thử nghiệm với 50 epochs)
print("--- Đang huấn luyện... ---")
auto_pet.fit(train_data, epochs=50)

# Lưu mô hình
auto_pet.save('pet_auto.h5')
print("--- ĐÃ XỬ LÝ XONG CÂU 2 ---")

--- Đang tải dữ liệu Cat vs Dog... ---
--- Đang huấn luyện... ---
Epoch 1/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 45s 284ms/step - loss: 0.6687
Epoch 2/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 37s 250ms/step - loss: 0.6522
Epoch 3/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 36s 244ms/step - loss: 0.6456
Epoch 4/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 38s 256ms/step - loss: 0.6348
Epoch 5/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 36s 245ms/step - loss: 0.6261
Epoch 6/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 41s 244ms/step - loss: 0.6220
Epoch 7/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 35s 238ms/step - loss: 0.6203
Epoch 8/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 36s 245ms/step - loss: 0.6179
Epoch 9/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 35s 239ms/step - loss: 0.6168
Epoch 10/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 42s 249ms/step - loss: 0.6151
Epoch 11/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 35s 236ms/step - loss: 0.6136
Epoch 12/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 41s 236ms/step - loss: 0.6131
Epoch 13/50
146/146 ━━━━━━━━━━━━━━━━━━━━ 36s 245ms/step - loss: 0.6129
Epoch 14/50
146/146 

--- ĐÃ XỬ LÝ XONG CÂU 2 ---


In [11]:
#câu 3
from keras.datasets import fashion_mnist

(x_train, _), (x_test, _) = fashion_mnist.load_data()
x_train = x_train.astype('float32') / 255.
x_train = x_train.reshape((len(x_train), 784))

# Cải tiến với L1 Regularizer như tài liệu hướng dẫn
input_img = keras.Input(shape=(784,))
encoded = keras.layers.Dense(32, activation='relu',
                             activity_regularizer=keras.regularizers.l1(10e-5))(input_img)
decoded = keras.layers.Dense(784, activation='sigmoid')(encoded)

auto_fashion = keras.Model(input_img, decoded)
auto_fashion.compile(optimizer='adam', loss='binary_crossentropy')
auto_fashion.fit(x_train, x_train, epochs=100, batch_size=256)

Epoch 1/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - loss: 0.6707
Epoch 2/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 0.6285
Epoch 3/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - loss: 0.5988
Epoch 4/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 0.5764
Epoch 5/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.5593
Epoch 6/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - loss: 0.5460
Epoch 7/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.5356
Epoch 8/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.5274
Epoch 9/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.5208
Epoch 10/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.5156
Epoch 11/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - loss: 0.5113
Epoch 12/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.5079
Epoch 13/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.5051
Epoch 14/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - loss: 0.5027
Epoch 15/100
235/235 ━━━━━━━━

In [12]:
#câu 4
from sklearn.datasets import fetch_lfw_people

# Tải dữ liệu khuôn mặt người (resize về 0.5 để nhẹ máy)
lfw_people = fetch_lfw_people(min_faces_per_person=70, resize=0.4)
x_faces = lfw_people.images
n_samples, h, w = x_faces.shape
x_faces = x_faces.reshape((n_samples, h * w)) / 255.0

input_img = keras.Input(shape=(h * w,))
encoded = keras.layers.Dense(128, activation='relu')(input_img)
decoded = keras.layers.Dense(h * w, activation='sigmoid')(encoded)

auto_face = keras.Model(input_img, decoded)
auto_face.compile(optimizer='adam', loss='binary_crossentropy')
auto_face.fit(x_faces, x_faces, epochs=200, batch_size=32)

Epoch 1/200
41/41 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 0.6431
Epoch 2/200
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.2974
Epoch 3/200
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0645
Epoch 4/200
41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 0.0278
Epoch 5/200
41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.0200
Epoch 6/200
41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 0.0171
Epoch 7/200
41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.0159
Epoch 8/200
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0154
Epoch 9/200
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0151
Epoch 10/200
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0150
Epoch 11/200
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0150
Epoch 12/200
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0149
Epoch 13/200
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0149
Epoch 14/200
41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0149
Epoch 15/200
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - lo

In [5]:
#câu 5
import os
import io
import base64
import numpy as np
import tensorflow as tf
from flask import Flask, request, render_template_string
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from PIL import Image

# ==========================================
# CẤU HÌNH VÀ NẠP MÔ HÌNH
# ==========================================
app = Flask(__name__)

# Tên file mô hình bạn đã lưu từ Câu 1
MODEL_PATH = 'cifar10_auto.h5'

try:
    # Nạp mô hình Autoencoder đã có [cite: 60]
    autoencoder = tf.keras.models.load_model(MODEL_PATH)
    print(f"--- Đã nạp thành công mô hình: {MODEL_PATH} ---")
except Exception as e:
    print(f"--- LỖI: Không tìm thấy file {MODEL_PATH}. Hãy kiểm tra lại đường dẫn! ---")

# ==========================================
# GIAI DIỆN HTML (INLINE TEMPLATE)
# ==========================================
html_template = '''
<!DOCTYPE html>
<html>
<head>
    <title>HUIT - Autoencoder Web App</title>
    <style>
        body { font-family: 'Segoe UI', Tahoma, sans-serif; text-align: center; background: #f0f2f5; padding: 50px; }
        .container { background: white; padding: 30px; border-radius: 20px; display: inline-block; box-shadow: 0 10px 30px rgba(0,0,0,0.1); }
        .upload-box { border: 2px dashed #3498db; padding: 20px; border-radius: 10px; margin-bottom: 20px; background: #ebf5fb; }
        .result-flex { display: flex; gap: 30px; justify-content: center; margin-top: 25px; }
        .img-card { text-align: center; }
        img { width: 160px; height: 160px; border-radius: 10px; border: 1px solid #ddd; image-rendering: pixelated; }
        .btn { background: #3498db; color: white; border: none; padding: 12px 25px; border-radius: 5px; cursor: pointer; font-weight: bold; }
        .btn:hover { background: #2980b9; }
        .formula { margin-top: 20px; font-style: italic; color: #7f8c8d; }
    </style>
</head>
<body>
    <div class="container">
        <h2>Hệ thống Tái tạo ảnh Autoencoder</h2>
        <p>Triển khai mô hình Câu 1 (CIFAR-10)</p>

        <form action="/predict" method="post" enctype="multipart/form-data" class="upload-box">
            <input type="file" name="file" accept="image/*" required><br><br>
            <button type="submit" class="btn">Xử lý ảnh qua Encoder-Decoder</button>
        </form>

        {% if original %}
        <div class="result-flex">
            <div class="img-card">
                <img src="data:image/png;base64,{{ original }}">
                <p><b>Ảnh gốc (x)</b></p>
            </div>
            <div style="align-self: center; font-size: 30px; color: #bdc3c7;">➔</div>
            <div class="img-card">
                <img src="data:image/png;base64,{{ reconstructed }}">
                <p><b>Ảnh tái tạo (o)</b></p>
            </div>
        </div>
        <p class="formula">Nguyên lý: o = D(E(x))</p>
        {% endif %}
    </div>
</body>
</html>
'''

# ==========================================
# LOGIC XỬ LÝ ẢNH
# ==========================================
def process_to_base64(img_arr):
    """Chuyển mảng ảnh sang chuỗi Base64 để hiển thị trên Web"""
    img = Image.fromarray(img_arr)
    buffered = io.BytesIO()
    img.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode()

@app.route('/')
def index():
    return render_template_string(html_template)

@app.route('/predict', methods=['POST'])
def predict():
    file = request.files.get('file')
    if not file: return "Vui lòng chọn ảnh!"

    # 1. Tiền xử lý ảnh (Resize về 32x32 theo CIFAR-10) [cite: 91]
    img = Image.open(io.BytesIO(file.read())).convert('RGB').resize((32, 32))
    img_arr = np.array(img).astype('float32') / 255.0

    # 2. Dự báo tái tạo (Flatten -> Predict -> Reshape) [cite: 13, 81]
    img_input = img_arr.reshape(1, 3072)
    reconstructed = autoencoder.predict(img_input)

    # Chuyển kết quả về dạng ảnh (32, 32, 3)
    res_arr = (reconstructed.reshape(32, 32, 3) * 255).astype('uint8')

    # 3. Trả về kết quả hiển thị
    return render_template_string(
        html_template,
        original=process_to_base64((img_arr * 255).astype('uint8')),
        reconstructed=process_to_base64(res_arr)
    )

if __name__ == '__main__':
    # Hỗ trợ chạy trên Google Colab
    try:
        from google.colab.output import eval_js
        print(f"Truy cập Web tại: {eval_js('google.colab.kernel.proxyPort(5000)')}")
    except:
        print("Chạy ứng dụng tại: http://127.0.0.1:5000")

    app.run(port=5000)

--- Đã nạp thành công mô hình: cifar10_auto.h5 ---
Truy cập Web tại: https://5000-m-s-kkb-usw3b0-2ull5dvlfwpf2-b.us-west3-0.prod.colab.dev
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [23/May/2026 01:03:04] "GET / HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step


INFO:werkzeug:127.0.0.1 - - [23/May/2026 01:03:09] "POST /predict HTTP/1.1" 200 -
